# DAG Planner RL Training Pipeline

All-in-one notebook: collect scored data via DeepSeek, train a LoRA adapter, output artifacts.

**Setup:** Kaggle notebook with T4 GPU, Internet ON.

**Phase 1:** Collect -- DeepSeek generates N candidate DAGs per puzzle, each executed via deterministic tools, scored against ground truth.

**Phase 2:** Train -- SFT on winning plans using Qwen-7B + QLoRA.

**Phase 3:** Output -- LoRA adapter + updated PLANNER_SYSTEM with few-shot examples.

## Phase 1: Setup

In [ ]:
!pip install -q trl peft bitsandbytes accelerate transformers datasets openai python-dotenv

In [ ]:
import sys, os, json, time, zipfile, shutil

# Unzip the uploaded code dataset
CODE_DATASET = "/kaggle/input/nemotron-pipeline-code"
WORK = "/kaggle/working"
CODE_DIR = os.path.join(WORK, "nemotron")

os.makedirs(CODE_DIR, exist_ok=True)
for zf in ["src.zip", "data.zip"]:
    zpath = os.path.join(CODE_DATASET, zf)
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(CODE_DIR)
        print(f"Extracted {zf}")

# Also check for unzipped files (Kaggle sometimes auto-extracts)
for folder in ["src", "data"]:
    src_path = os.path.join(CODE_DATASET, folder)
    dst_path = os.path.join(CODE_DIR, folder)
    if os.path.isdir(src_path) and not os.path.isdir(dst_path):
        shutil.copytree(src_path, dst_path)
        print(f"Copied {folder}/")

sys.path.insert(0, CODE_DIR)
print("Files:", os.listdir(CODE_DIR))
print("src/:", os.listdir(os.path.join(CODE_DIR, "src")))

In [ ]:
# Configure DeepSeek -- use Kaggle Secrets or paste key here
try:
    from kaggle_secrets import UserSecretsClient
    DEEPSEEK_API_KEY = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
    print("Loaded DeepSeek key from Kaggle Secrets")
except Exception:
    DEEPSEEK_API_KEY = ""  # paste your key here if not using Secrets
    print("Using hardcoded DeepSeek key")

assert DEEPSEEK_API_KEY, "Set DEEPSEEK_API_KEY in Kaggle Secrets or paste it above"

# Patch environment so src.config picks it up
os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY
os.environ["LLM_PROVIDER"] = "deepseek"
os.environ["TRAIN_PATH"] = os.path.join(CODE_DIR, "data", "train.csv")

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Import project modules
from src.llm_client import LLMClient
from src.planner import PLANNER_SYSTEM, _parse_planner_output, _build_dag
from src.solver import _solve_single_node, _find_sink, _extract_final_answer
from src.state import ThoughtNode
from src.classify import PUZZLE_SIGNATURES
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

print(f"PLANNER_SYSTEM: {len(PLANNER_SYSTEM)} chars")
print("Tools imported OK")

## Phase 2: Collect Scored Data

In [ ]:
# -- Configuration --
N_CANDIDATES = 4          # DAG candidates per puzzle
PUZZLE_LIMIT = 200        # max puzzles to process
PUZZLE_TYPES = [          # which types to collect (comment out to include all)
    "bit_manipulation",
    "gravity_physics",
    "unit_conversion",
    "numeral_conversion",
    # "cipher_decryption",   # slow: multi-node DAG with LLM execution
    # "equation_transform",  # slow: LLM fallback on solver failure
]
MIN_REWARD = 0.5          # threshold for "winner" training examples

In [ ]:
# -- Helper functions (same as train_planner.py) --

def compute_reward(expected, got, dag_valid):
    if not dag_valid:
        return -1.0
    a, b = got.strip(), expected.strip()
    if a == b:
        return 1.0
    try:
        if abs(float(a) - float(b)) <= 1e-2 + 1e-9:
            return 0.5
    except (ValueError, TypeError):
        pass
    return -0.5

def detect_type(prompt):
    lower = prompt.lower()
    for sig, ptype in PUZZLE_SIGNATURES.items():
        if sig in lower:
            return ptype
    return "unknown"

def execute_dag(dag, llm, prompt):
    max_rounds = len(dag) + 2
    for _ in range(max_rounds):
        answered_ids = {n["id"] for n in dag if n["answer"] is not None}
        ready = [
            n for n in dag
            if n["answer"] is None
            and all(p in answered_ids for p in n["depends_on"])
        ]
        if not ready:
            break
        with ThreadPoolExecutor(max_workers=max(len(ready), 1)) as pool:
            futs = {pool.submit(_solve_single_node, llm, n, dag, prompt): n for n in ready}
            for fut in as_completed(futs):
                node = futs[fut]
                try:
                    ans = fut.result(timeout=120)
                except Exception:
                    ans = ""
                for n in dag:
                    if n["id"] == node["id"]:
                        n["answer"] = ans
                        break
    sink = _find_sink(dag)
    raw = sink.get("answer") or ""
    return _extract_final_answer(raw) or raw

def generate_candidate(planner_llm, puzzle_type, prompt, temperature):
    resp = planner_llm.chat(
        [{"role": "system", "content": PLANNER_SYSTEM},
         {"role": "user", "content": f"PUZZLE_TYPE: {puzzle_type}\n\nPROMPT:\n{prompt}"}],
        think=False, temperature=temperature, max_tokens=8192,
    )
    raw = (resp.content or "").strip()
    if not raw:
        return "", None
    try:
        edges, nodes_dict = _parse_planner_output(raw)
        dag = _build_dag(edges, nodes_dict, prompt)
        return raw, dag
    except Exception:
        return raw, None

print("Helpers defined")

In [ ]:
# -- Run data collection --

planner_llm = LLMClient(
    provider="deepseek",
    deepseek_api_key=DEEPSEEK_API_KEY,
    deepseek_model="deepseek-chat",
)
exec_llm = LLMClient(
    provider="deepseek",
    deepseek_api_key=DEEPSEEK_API_KEY,
    deepseek_model="deepseek-chat",
)

train_path = os.path.join(CODE_DIR, "data", "train.csv")
df = pd.read_csv(train_path)
df["puzzle_type"] = df["prompt"].apply(detect_type)
if PUZZLE_TYPES:
    df = df[df["puzzle_type"].isin(PUZZLE_TYPES)]
df = df.head(PUZZLE_LIMIT)

step = 0.7 / max(N_CANDIDATES - 1, 1)
temps = [round(0.2 + i * step, 2) for i in range(N_CANDIDATES)]

print(f"Puzzles: {len(df)}  |  Candidates/puzzle: {N_CANDIDATES}  |  Temps: {temps}")

all_records = []
stats = {"total": 0, "exact": 0, "valid": 0, "reward_sum": 0.0}

for idx, (_, row) in enumerate(df.iterrows(), 1):
    row_id = row["id"]
    prompt = row["prompt"]
    expected = str(row.get("answer", ""))
    puzzle_type = row["puzzle_type"]
    
    print(f"[{idx}/{len(df)}] {puzzle_type} {row_id} expected={expected!r}")
    
    for ci, temp in enumerate(temps):
        t0 = time.time()
        raw_plan, dag = generate_candidate(planner_llm, puzzle_type, prompt, temp)
        dag_valid = dag is not None
        
        got = ""
        if dag_valid:
            try:
                got = execute_dag(dag, exec_llm, prompt)
            except Exception:
                got = ""
        
        reward = compute_reward(expected, got, dag_valid)
        elapsed = round(time.time() - t0, 1)
        
        record = {
            "puzzle_id": row_id,
            "puzzle_type": puzzle_type,
            "candidate": ci,
            "temperature": temp,
            "prompt": prompt,
            "planner_output": raw_plan[:4000],
            "dag_valid": dag_valid,
            "dag_nodes": len(dag) if dag else 0,
            "got": got,
            "expected": expected,
            "reward": reward,
            "elapsed_s": elapsed,
        }
        all_records.append(record)
        
        stats["total"] += 1
        stats["reward_sum"] += reward
        if reward >= 1.0: stats["exact"] += 1
        if dag_valid: stats["valid"] += 1
        
        tag = "OK" if reward >= 0.5 else ("VALID" if dag_valid else "BAD_DAG")
        print(f"  c{ci} T={temp} {tag} got={got!r} reward={reward} ({elapsed}s)")

# Save scored data
scores_path = os.path.join(WORK, "planner_scores.jsonl")
with open(scores_path, "w", encoding="utf-8") as f:
    for r in all_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

n = stats["total"] or 1
print(f"\n=== Collection Summary ===")
print(f"Total samples:  {stats['total']}")
print(f"Valid DAGs:     {stats['valid']} ({100*stats['valid']/n:.1f}%)")
print(f"Exact matches:  {stats['exact']} ({100*stats['exact']/n:.1f}%)")
print(f"Mean reward:    {stats['reward_sum']/n:.3f}")
print(f"Saved to:       {scores_path}")

## Phase 3: Train LoRA Adapter

In [ ]:
# -- Build training dataset from winners --

winners = [r for r in all_records if r["reward"] >= MIN_REWARD and r["dag_valid"]]
print(f"Winners: {len(winners)} / {len(all_records)}")

# Deduplicate: keep best candidate per puzzle
best_per_puzzle = {}
for r in winners:
    pid = r["puzzle_id"]
    if pid not in best_per_puzzle or r["reward"] > best_per_puzzle[pid]["reward"]:
        best_per_puzzle[pid] = r

from collections import Counter
print(f"Unique puzzles with wins: {len(best_per_puzzle)}")
print("By type:", Counter(r["puzzle_type"] for r in best_per_puzzle.values()))

if len(best_per_puzzle) < 5:
    print("WARNING: Very few training examples. Consider collecting more data.")

In [ ]:
from datasets import Dataset

def build_chat(record):
    user_msg = f"PUZZLE_TYPE: {record['puzzle_type']}\n\nPROMPT:\n{record['prompt']}"
    return {
        "messages": [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": record["planner_output"]},
        ]
    }

train_data = [build_chat(r) for r in best_per_puzzle.values()]
dataset = Dataset.from_list(train_data)
print(f"Training dataset: {len(dataset)} examples")

In [ ]:
# -- Load base model (4-bit quantized) --

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"Model loaded: {BASE_MODEL}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# -- Apply LoRA --

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# -- Train --

from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = os.path.join(WORK, "planner-lora")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    max_seq_length=2048,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# -- Save adapter --

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

total_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, f))
)
print(f"Adapter size: {total_size/1e6:.1f} MB")

## Phase 4: Output Artifacts

In [ ]:
# -- Generate updated PLANNER_SYSTEM with few-shot examples --

# Pick best examples per type for few-shot
fewshot_by_type = {}
for r in best_per_puzzle.values():
    pt = r["puzzle_type"]
    if pt not in fewshot_by_type or r["reward"] > fewshot_by_type[pt]["reward"]:
        fewshot_by_type[pt] = r

fewshot_section = "\n\nFEW-SHOT EXAMPLES OF CORRECT PLANS:\n"
for pt, r in sorted(fewshot_by_type.items()):
    fewshot_section += f"\n=== {pt} (verified correct, reward={r['reward']}) ===\n"
    fewshot_section += f"Input: PUZZLE_TYPE: {pt}\n"
    fewshot_section += f"Output:\n{r['planner_output'][:1500]}\n"

# Insert few-shot examples before the RULES section
updated_prompt = PLANNER_SYSTEM.replace(
    "RULES:\n",
    fewshot_section + "\nRULES:\n"
)

prompt_path = os.path.join(WORK, "planner_system_updated.txt")
with open(prompt_path, "w", encoding="utf-8") as f:
    f.write(updated_prompt)

print(f"Updated PLANNER_SYSTEM: {len(updated_prompt)} chars (was {len(PLANNER_SYSTEM)})")
print(f"Saved to: {prompt_path}")
print(f"\nFew-shot types: {list(fewshot_by_type.keys())}")

In [ ]:
# -- Summary of output artifacts --

print("=" * 60)
print("OUTPUT ARTIFACTS (download from notebook output)")
print("=" * 60)
print(f"1. LoRA adapter:          {OUTPUT_DIR}/")
print(f"2. Updated PLANNER_SYSTEM: {prompt_path}")
print(f"3. Scored data:           {scores_path}")
print()
print("To sync back to local:")
print("  kaggle kernels output tianlinzhao1/grpo-planner -p kaggle_output/")
print("  Then copy planner_system_updated.txt content into PLANNER_SYSTEM in src/planner.py")

In [ ]:
# -- Quick sanity check: generate a plan with the trained model --

test_prompt = """PUZZLE_TYPE: gravity_physics

PROMPT:
In Alice's Wonderland, the gravitational constant has been secretly changed. Here are some example observations:
For t = 1.37s, distance = 14.92 m
For t = 4.27s, distance = 144.96 m
For t = 3.28s, distance = 85.54 m
Now, determine the falling distance for t = 4.41s given d = 0.5*g*t^2."""

messages = [
    {"role": "system", "content": PLANNER_SYSTEM},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
with torch.no_grad():
    output = model.generate(inputs, max_new_tokens=512, temperature=0.3, do_sample=True)

generated = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
print("Generated plan (trained model):")
print(generated)